# Creating a Heatmap of Wildfire occurences using Folium Package

In [1]:
%pip install folium geopandas pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 323 kB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 19.5 MB 647 kB/s eta 0:00:01
     |████████████████████████████████| 4.9 MB 16.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import geopandas as gpd
import folium

/Users/maliyer/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
df = pd.read_csv("all_fires_full.csv")

df.head()

,fire_id,date,days_to_ignition,phase,bi,erc,fm100,fm1000,tmmx,tmmn,...,nlcd_forest_ever,nlcd_forest_mixed,nlcd_shrub,nlcd_grassland,nlcd_pasture,nlcd_crops,nlcd_wetland_woody,nlcd_wetland_herb,tmmx_c,tmmn_c
0,10TH_2024_00139,2023-06-25,-365,pre_fire,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.992593,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,10TH_2024_00139,2023-06-26,-364,pre_fire,93.666664,83.666664,5.233334,7.183334,301.60000,289.84998,...,0.0,0.0,0.992593,0.0,0.0,0.0,0.0,0.0,28.45000,16.69998
2,10TH_2024_00139,2023-06-27,-363,pre_fire,89.500000,84.000000,5.350000,7.100000,305.19998,291.13333,...,0.0,0.0,0.992593,0.0,0.0,0.0,0.0,0.0,32.04998,17.98333
3,10TH_2024_00139,2023-06-28,-362,pre_fire,85.833336,84.333336,5.400000,6.983334,304.85000,291.20000,...,0.0,0.0,0.992593,0.0,0.0,0.0,0.0,0.0,31.70000,18.05000
4,10TH_2024_00139,2023-06-29,-361,pre_fire,62.166668,86.333336,5.250000,6.883334,309.86667,291.40000,...,0.0,0.0,0.992593,0.0,0.0,0.0,0.0,0.0,36.71667,18.25000


In [17]:
df["fire_id"].nunique(), len(df)

(7662, 2872673)

In [19]:
fires = df.groupby("fire_id").agg({
    "lat_centroid": "first",
    "lon_centroid": "first"
}).reset_index()

fires.head()

,fire_id,lat_centroid,lon_centroid
0,10TH_2024_00139,34.971166,-118.109886
1,10_2003_07027,40.132079,-124.084808
2,118_FWY_2015_03742,34.275892,-118.625533
3,128_2002_07316,38.507163,-122.091515
4,128_2006_06210,38.438790,-122.169162


In [21]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    fires,
    geometry=gpd.points_from_xy(fires.lon_centroid, fires.lat_centroid),
    crs="EPSG:4326"
)

In [5]:
url = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/california-counties.geojson"

counties = gpd.read_file(url)

counties.head()

,name,cartodb_id,created_at,updated_at,geometry
0,Alameda,1,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00,"MULTIPOLYGON (((-122.31293 37.89733, -122.2884..."
1,Alpine,2,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00,"POLYGON ((-120.07239 38.70277, -119.96495 38.7..."
2,Amador,3,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00,"POLYGON ((-121.02726 38.48925, -121.02741 38.5..."
3,Butte,4,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00,"POLYGON ((-121.87925 39.30361, -121.90831 39.3..."
4,Calaveras,5,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00,"POLYGON ((-120.87605 38.02889, -120.91875 38.0..."


In [6]:
counties = counties.to_crs("EPSG:4326")

In [22]:
fires_with_county = gpd.sjoin(gdf, counties, how="inner", predicate="within")
fires_with_county.head()

,fire_id,lat_centroid,lon_centroid,geometry,index_right,name,cartodb_id,created_at,updated_at
0,10TH_2024_00139,34.971166,-118.109886,POINT (-118.10989 34.97117),36,Kern,15,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00
1,10_2003_07027,40.132079,-124.084808,POINT (-124.08481 40.13208),12,Humboldt,12,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00
2,118_FWY_2015_03742,34.275892,-118.625533,POINT (-118.62553 34.27589),21,Los Angeles,19,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00
3,128_2002_07316,38.507163,-122.091515,POINT (-122.09151 38.50716),19,Solano,48,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00
4,128_2006_06210,38.438790,-122.169162,POINT (-122.16916 38.43879),47,Napa,28,2015-07-04 21:04:58+00:00,2015-07-04 21:04:58+00:00


In [23]:
county_counts = fires_with_county.groupby("name").size().reset_index(name="fire_count")

county_counts.head()

,name,fire_count
0,Alameda,33
1,Alpine,19
2,Amador,43
3,Butte,136
4,Calaveras,93


In [24]:
county_map_data = counties.merge(county_counts, on="name", how="left")

county_map_data["fire_count"] = county_map_data["fire_count"].fillna(0)

In [28]:
m = folium.Map(
    location=[37, -120],
    zoom_start=6,
    tiles="cartodbpositron"
)

In [29]:
def style_function(feature):
    fire_count = feature["properties"]["fire_count"]

    if fire_count > 500:
        color = "#800026"
    elif fire_count > 200:
        color = "#BD0026"
    elif fire_count > 100:
        color = "#E31A1C"
    elif fire_count > 50:
        color = "#FC4E2A"
    elif fire_count > 20:
        color = "#FD8D3C"
    else:
        color = "#FEB24C"

    return {
        "fillColor": color,
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.7,
    }

In [31]:
# Keep only the columns needed for the map
map_data = county_map_data[["name", "fire_count", "geometry"]].copy()

# Convert wildfire counts to regular integers (avoids serialization issues)
map_data["fire_count"] = map_data["fire_count"].astype(int)

folium.GeoJson(
    map_data.to_json(),
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "fire_count"],
        aliases=["County:", "Wildfire Count:"],
        localize=True
    ),
    popup=folium.GeoJsonPopup(
        fields=["name", "fire_count"],
        aliases=["County:", "Number of Wildfires:"]
    )
).add_to(m)

In [26]:
# Keep only the columns folium needs
choropleth_data = county_map_data[["name", "fire_count", "geometry"]]

folium.Choropleth(
    geo_data=choropleth_data.to_json(),   # convert safely to json
    data=choropleth_data,
    columns=["name", "fire_count"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Wildfire Occurrences by County"
).add_to(m)

In [32]:
m.save("california_wildfires_map.html")